Optimizing tool selection...I'll create a comprehensive summary of the entire Streamlit app with all features and their relationships.

## 📊 **Streamlit N-BEATS Dashboard - Complete Architecture Summary**

---

### **1. PROJECT STRUCTURE**



In [ ]:
streamlit_app/
├── app.py                 # Main dashboard application
├── model_loader.py        # Model registry & loading logic
├── data_loader.py         # Data filtering & preparation
├── inference.py           # Model prediction pipeline
├── visualization.py       # Chart generation (Plotly)
├── seasonal_blending.py   # Schedule-based model blending
└── logs/                  # Streamlit session logging



---

### **2. CORE MODULE BREAKDOWN**

#### **A. model_loader.py — Model Discovery & Loading**

**Key Classes & Functions:**

| Function | Purpose |
|----------|---------|
| `load_registry()` | Load single model_registry.json |
| `load_all_registries()` | **NEW**: Load & merge all zone registries + auto-discover models |
| `get_available_zones()` | Extract unique zones from loaded models |
| `list_models()` | Return sorted model names |
| `model_display_label()` | **NEW**: Generate clean readable labels (e.g., "TPCODL \| 1L × 512W \| 7d ctx \| Jan, Dec \| MAPE 3.21%") |
| `load_model()` | Load Darts checkpoint with correct `work_dir` path |
| `load_scaler()` | Load joblib scaler for inverse transforms |
| `discover_models_in_directory()` | Recursively scan for models in subdirs (ZONE/models/) |
| `_find_work_dir()` | Walk up directory tree to locate correct darts_logs folder |

**Key Improvements:**
- ✅ Supports both flat (root models) and zone-based (models, etc.) layouts
- ✅ Auto-discovers zone registries on startup
- ✅ Sets `work_dir` on each model entry so Darts loads from correct checkpoint path
- ✅ Appends darts_logs to work_dir when passing to Darts (Darts doesn't auto-append when work_dir is explicit)

---

#### **B. `data_loader.py` — Data Filtering**

**Variables:**
- `AVAILABLE_ZONES`: All demand zones
- `MONTH_NAMES`: Month abbreviations (Jan, Feb, etc.)
- `TIME_COLUMN`: "Timestamp"

**Functions:**
| Function | Purpose |
|----------|---------|
| `load_raw_data()` | Load CSV, parse timestamps, rename columns |
| `get_date_bounds()` | Return min/max dates in data |
| `filter_data()` | Filter by zone, date range, optional months |

---

#### **C. `inference.py` — Predictions**

**Functions:**
| Function | Purpose |
|----------|---------|
| `run_inference()` | Run model on filtered series, return Timestamp + Actual + Predicted |
| `compute_metrics()` | Calculate MAPE, MAE, RMSE, R² |

---

#### **D. visualization.py — Charts**

**Functions:**
| Function | Purpose |
|----------|---------|
| `build_comparison_chart()` | Actual + multiple model predictions (Model Comparison mode) |
| `build_error_chart()` | Per-model error traces over time |
| `build_blending_chart()` | **NEW IMPROVED**: Actual + merged prediction + individual model traces **clipped to schedule segments** |
| `build_blending_error_chart()` | Error for merged blended prediction |

**Visualization Improvements:**
- ✅ Individual model traces now only show within their scheduled date ranges (not full dataset)
- ✅ Multiple schedule entries for same model grouped under one legend entry
- ✅ Fallback model shown across full range at lower opacity
- ✅ Segment background shading with model labels

---

#### **E. seasonal_blending.py — Schedule-Based Blending**

**Functions:**

| Function | Purpose |
|----------|---------|
| `validate_schedule()` | Check for overlaps, gaps, missing models |
| `apply_hard_merge()` | **NEW IMPROVED**: Merge predictions by timestep-level assignment |
| `apply_soft_transition()` | Hard merge + linear blending at segment boundaries |
| `compute_segment_metrics()` | Per-segment MAPE/MAE/RMSE/R² for each schedule entry |
| `compute_contribution_summary()` | Count how many timestamps each model contributed |

**Key Blending Logic Changes:**
- ✅ **Only timestamps within schedule segments are kept** (outside segments → blank, not fallback)
- ✅ Fallback model only fills NaN **inside** segments, not outside
- ✅ Later schedule entries override earlier ones on overlap
- ✅ Soft transition blends model predictions linearly at segment boundaries

---

### **3. DASHBOARD FEATURES — APP FLOW**

#### **SIDEBAR CONTROLS** (Always visible)



In [ ]:
📈 Dashboard Mode (Radio)
├── 📈 Model Comparison
└── 🌾 Seasonal Blending

⚙️ Evaluation Settings
├── Region / Zone (Selectbox)
├── Start Date (Date Input)
├── End Date (Date Input)
└── Months (Multiselect) [optional filter]

🧠 Model Selection
├── Filter models by zone (Selectbox) [**NEW**]
├── ✅ Select All / 🗑️ Clear buttons [**FIXED**]
└── Choose models multiselect (filtered by zone)



**Zone Filter Feature:**
- Dropdown shows all unique zones from registry
- Selecting a zone filters the model list below
- "All Zones" shows everything
- Select All / Clear All operate on **filtered** list only
- Stale selections cleaned up when filter changes

---

#### **MODE A: 📈 Model Comparison**

**Flow:**
1. Select models from filtered dropdown
2. Press "🚀 Run Evaluation"
3. **Output:**
   - Line chart: Actual (red) + each model prediction (dotted)
   - Error chart: Per-model prediction error over time
   - Metrics table: MAPE, MAE, RMSE, R² sorted by MAPE (best first)
   - Raw predictions expander

**Key Features:**
- ✅ Clean display labels (e.g., "TPCODL \| 1L × 512W \| 7d ctx")
- ✅ Zone filtering reduces dropdown clutter
- ✅ Select All / Clear All work on filtered subset

---

#### **MODE B: 🌾 Seasonal Blending**

**Workflow:**



In [ ]:
1. Select Models
   └── ✅ Select All / 🗑️ Clear buttons (filtered list)

2. Choose Fallback Model (for NaN gaps inside segments)

3. Transition Settings (optional)
   └── Enable soft transition + transition window (days)

4. Build Schedule
   ├── ➕ Add Schedule Entry expander
   │   ├── Segment start (date)
   │   ├── Segment end (date)
   │   └── Assign model (dropdown)
   │
   └── Current Schedule (displayed as rows)
       ├── Start | End | Model | 🗑️ (remove per-row)
       └── 🗑️ Clear All Entries button [**NEW UI**]

5. Press "🚀 Run Blending"



**Schedule Management Features:**
- ✅ Per-row trash buttons for instant removal (better UX than index input)
- ✅ "Clear All Entries" button to wipe schedule
- ✅ Validation warnings for overlaps, gaps, missing models
- ✅ Region-aware (schedule different per zone, persisted in session state)

**Blending Outputs:**
1. **Chart:** Actual (red) + Merged prediction (thick orange) + individual model traces **clipped to segments**
   - Segment background shading with model labels
   - Transition zones highlighted (dotted boundaries) if soft transition enabled
   
2. **Error Chart:** Merged prediction error

3. **Metrics:**
   - Overall merged metrics
   - Per-segment metrics (one row per schedule entry)
   - Model contribution summary (timestamps per source)

4. **Raw Data Expander:** Full merged dataframe with Source column

---

### **4. KEY DATA STRUCTURES**

#### **Model Registry Entry (JSON)**


In [ ]:
{
  "nbeats_TPCODL_...": {
    "zone": "TPCODL Demand",
    "config": {
      "num_layers": 1,
      "layer_widths": 256,
      "input_chunk_length": 672,
      "num_stacks": 30
    },
    "metrics": {"test_MAPE": 3.21},
    "checkpoint_dir": "TPCODL/models/nbeats_TPCODL_...",
    "scaler_path": "TPCODL/models/nbeats_TPCODL_.../scaler.joblib",
    "work_dir": "TPCODL",
    "zone": "TPCODL Demand"
  }
}



#### **Schedule Entry**


In [ ]:
{
  "start": "2025-01-01",
  "end": "2025-03-31",
  "model": "nbeats_TPCODL_1l_256w_ctx7d_..."
}



#### **Merged Prediction DataFrame**


In [ ]:
# Columns:
# - Timestamp
# - Actual (values from filtered data)
# - Predicted (merged from schedule + fallback)
# - Source (which model contributed)



---

### **5. CACHING STRATEGY**



In [ ]:
@st.cache_data
def _load_data():
    return load_raw_data()

@st.cache_data
def _load_registry():
    return load_all_registries()  # Auto-merges all zones

@st.cache_resource
def _load_model_cached(model_name, _entry_json):
    return load_model(), load_scaler()  # Cached by model_name



**Key:** `_entry_json` passed as string to ensure cache hits only when entry content matches.

---

### **6. SESSION STATE PERSISTENCE**



In [ ]:
st.session_state:
├── cmp_models              # Comparison mode selected labels
├── blend_models            # Blending mode selected labels
├── model_zone_filter       # Current zone filter selection
├── blend_schedules         # Dict[zone] = list of schedule entries
├── blend_soft              # Soft transition enabled?
├── blend_td                # Transition days (int)
├── show_indiv              # Show individual predictions in blend chart?
└── (date/range inputs auto-managed by Streamlit)



---

### **7. ERROR HANDLING & VALIDATION**

| Issue | Handling |
|-------|----------|
| No models selected | Info message + `st.stop()` |
| Model not in registry | Warning, skip to next |
| Empty filtered data | Error message + `st.stop()` |
| Model produces no predictions | Warning |
| Schedule gaps | Warning message (fallback will fill) |
| Schedule overlaps | Warning (later entry wins) |
| Stale model selections after zone filter | Auto-cleaned from session state |
| Date range invalid | Error (start >= end) |
| Fallback model produces no predictions | Error + `st.stop()` |

---

### **8. DIRECTORY STRUCTURE SUPPORT**

**Two layouts now supported:**

**Layout A (Root Level):**


In [ ]:
models/
├── model_registry.json
├── nbeats_TPNODL_...
│   └── scaler.joblib
└── nbeats_TOTAL_...
    └── scaler.joblib

darts_logs/
├── nbeats_TPNODL_.../checkpoints/
└── nbeats_TOTAL_.../checkpoints/



**Layout B (Zone-Based):**


In [ ]:
TPCODL/
├── models/
│   ├── model_registry.json
│   └── nbeats_TPCODL_...
│       └── scaler.joblib
└── darts_logs/
    └── nbeats_TPCODL_.../checkpoints/

TPWODL/
├── models/
│   └── ...
└── darts_logs/
    └── ...



**On startup:** `load_all_registries()` finds and merges all zone registries + auto-discovers any orphaned models.

---

### **9. FEATURE DELTA SUMMARY**

**Original Issues → Solutions:**

| Issue | Solution |
|-------|----------|
| ❌ Models only from root | ✅ Recursively scan all zones + auto-merge registries |
| ❌ Model names raw & unreadable | ✅ Clean labels: "TPCODL \| 1L × 512W \| 7d ctx" |
| ❌ Select All/Clear broken | ✅ Direct session_state binding (no separate default key) |
| ❌ Hard to remove schedule entries | ✅ Per-row trash buttons + Clear All |
| ❌ Individual model traces span full range | ✅ Clipped to schedule segments only |
| ❌ Fallback fills all gaps | ✅ Only fills NaN inside segments; outside = blank |
| ❌ Zone filter missing | ✅ Dropdown to filter model list |
| ❌ Darts path construction wrong | ✅ Append darts_logs to explicit work_dir |

---

### **10. USER WORKFLOWS**

**Workflow A: Quick Comparison**
1. Sidebar: Zone → Select models → Run Evaluation
2. See chart + metrics
3. Done

**Workflow B: Seasonal Blending**
1. Sidebar: Zone → Select models
2. Main: Add schedule entry (Jan 1 - Apr 30 → Model A)
3. Main: Add schedule entry (May 1 - Dec 31 → Model B)
4. Choose fallback model
5. Run Blending
6. See merged chart + per-segment metrics

**Workflow C: Zone Comparison**
1. Sidebar: Zone filter → TPCODL
2. See only TPCODL models in dropdown
3. Select models → Compare
4. Switch zone filter to TPWODL
5. (Dropdown refreshes, old selections cleared)

---

### **11. PERFORMANCE NOTES**

- ✅ Model checkpoints cached by `@st.cache_resource` (loaded once per session)
- ✅ Registry & data cached by `@st.cache_data` (survives reruns)
- ✅ Charts downsampled to 4000 points max for smooth interaction
- ✅ No data reloaded on zone filter change (only UI updated)

---

This maps every feature, every file, every UI component, and every data flow in the app. Each piece works together to enable both quick model comparisons and sophisticated seasonal blending workflows! 🚀